# Notebook 05: Comparative Evaluation and Statistical Analysis

**Course:** DS 4889 Machine Learning, United International University

**Course outcomes:** CO5 (sound validation and interpretation), CO6 (translating predictions into actionable decisions), CO7 (unsupervised pattern discovery), CO8 (comparison of alternative approaches)

**Lecture mapping:** Lecture 02 (evaluation metrics, cross validation and statistical comparison), Lectures 03 and 05 (logistic regression and maximum likelihood), Lecture 06 (Gaussian Mixture Models and Principal Component Analysis)

## Objective

The preceding notebooks produced three families of experiments: a convolutional baseline trained on
preprocessed images, an architecturally identical convolutional model trained on Gaussian Mixture
attention weighted images, and a set of classical classifiers trained on engineered descriptors. Each
was evaluated in isolation. This notebook consolidates them into the single set of tables and figures
reported for the project, and applies the statistical machinery required to decide whether the observed
differences are real.

Three questions are answered here. First, which pipeline performs best on the held out test set, and
by how much. Second, whether the difference between the two convolutional pipelines exceeds what
could be produced by sampling variation alone, which is established with a paired significance test
rather than by comparing point estimates. Third, whether the better performing pipeline is the one
that should be deployed, which additionally requires its computational cost at inference time to be
taken into account.


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import auc as sklearn_auc
from sklearn.metrics import confusion_matrix, roc_curve
from sklearn.preprocessing import label_binarize

sys.path.append('..')
from src import (
    config,
    data_loader,
    preprocessing,
    gmm_segmentation,
    classical_models,
    cnn_model,
    evaluate,
)

cnn_model.set_global_seed()
sns.set_theme(style='whitegrid', context='notebook')

print('TensorFlow version:', tf.__version__)

# The test split is loaded exactly as it was in notebooks 03 and 04, in the same sorted file
# order, so that predictions from the two models can be paired sample by sample.
x_test_raw, y_test, test_paths = data_loader.load_split(
    config.TEST_DIR,
    image_size=config.IMAGE_SIZE,
    grayscale=False,
    limit_per_class=None,
    show_progress=True,
)

x_test = preprocessing.preprocess_batch(x_test_raw, image_size=config.IMAGE_SIZE)

print('Test images:', x_test.shape)
display(data_loader.class_distribution(y_test))


## Loading the trained models

Both convolutional models are read from disk rather than retrained. Retraining inside this notebook
would be wasteful and, more importantly, would risk producing numbers that differ from those reported
in notebooks 03 and 04, since any divergence in the execution environment would propagate into the
weights. Reading the stored artefacts guarantees that the figures consolidated here are the same
figures that were generated when each model was trained and validated.

The hybrid model requires its own test tensor, because it was trained on attention weighted inputs
and presenting it with unweighted images would evaluate it outside the distribution it was fitted on.
That tensor is recovered from the cache written by notebook 04 when the cache exists and its shape
matches the current test split, and is recomputed from scratch otherwise. The recomputation path is
retained so that the notebook remains runnable on a clean checkout, at the cost of a per image mixture
fit.


In [ ]:
baseline_path = config.MODELS_DIR / config.BASELINE_MODEL_NAME
hybrid_path = config.MODELS_DIR / config.HYBRID_MODEL_NAME

baseline_model = None
hybrid_model = None

try:
    baseline_model = tf.keras.models.load_model(baseline_path)
    print('Loaded baseline model from', baseline_path)
except Exception as error:
    print('Could not load the baseline model:', type(error).__name__, error)
    print('Run notebook 03 (03_cnn_baseline.ipynb) from top to bottom before running this notebook.')

try:
    hybrid_model = tf.keras.models.load_model(hybrid_path)
    print('Loaded hybrid model from', hybrid_path)
except Exception as error:
    print('Could not load the hybrid model:', type(error).__name__, error)
    print('Run notebook 04 (04_hybrid_gmm_cnn.ipynb) from top to bottom before running this notebook.')

if baseline_model is None or hybrid_model is None:
    raise RuntimeError(
        'Both trained models are required. Execute notebooks 03 and 04 first, then rerun this cell.'
    )

cache_path = config.CACHE_DIR / 'hybrid_arrays.npz'
x_test_hybrid = None

if cache_path.exists():
    with np.load(cache_path) as cached:
        if 'x_test' in cached.files and tuple(cached['x_test'].shape) == tuple(x_test.shape):
            x_test_hybrid = cached['x_test']
            print('Recovered the hybrid test tensor from', cache_path)
        else:
            print('Cache present but the stored test shape does not match; recomputing.')

if x_test_hybrid is None:
    recompute_start = time.time()
    x_test_hybrid = gmm_segmentation.transform_batch_for_hybrid(
        x_test, show_progress=True, description='GMM transform, test split'
    )
    print('Recomputed in {0:.1f} seconds.'.format(time.time() - recompute_start))

print('Hybrid test tensor:', x_test_hybrid.shape)


In [ ]:
baseline_probabilities = baseline_model.predict(x_test, batch_size=config.BATCH_SIZE, verbose=0)
baseline_predictions = np.argmax(baseline_probabilities, axis=1)

hybrid_probabilities = hybrid_model.predict(x_test_hybrid, batch_size=config.BATCH_SIZE, verbose=0)
hybrid_predictions = np.argmax(hybrid_probabilities, axis=1)

print('Baseline predictions:', baseline_predictions.shape)
print('Hybrid predictions:  ', hybrid_predictions.shape)


def measure_latency(model, images, n_images=100):
    """Return the mean single image forward pass time in milliseconds.

    Images are submitted one at a time rather than in batches because that is how a deployed
    system receives a scan. A short warm up is discarded first so that graph construction and
    kernel compilation are not charged to the measurement.
    """
    sample = images[:min(n_images, len(images))]
    for warmup_image in sample[:5]:
        model.predict(np.expand_dims(warmup_image, axis=0), verbose=0)

    start = time.time()
    for image in sample:
        model.predict(np.expand_dims(image, axis=0), verbose=0)
    elapsed = time.time() - start

    return 1000.0 * elapsed / len(sample)


baseline_latency_ms = measure_latency(baseline_model, x_test, n_images=100)
hybrid_network_latency_ms = measure_latency(hybrid_model, x_test_hybrid, n_images=100)

print('Baseline forward pass:        {0:.2f} ms per image'.format(baseline_latency_ms))
print('Hybrid forward pass:          {0:.2f} ms per image'.format(hybrid_network_latency_ms))
print('Note: the hybrid figure above excludes the segmentation stage, which is timed separately.')


## Classical feature baselines

A convolutional network is justified only if the problem actually requires representation learning. If
a logistic regression fitted to a handful of engineered descriptors matched the network, the
additional parameters, training time and opacity of the deep model would not be warranted. The
classical models therefore serve as the lower reference point of the study, in the same way that the
baseline convolutional network serves as the reference for the hybrid pipeline.

The descriptors combine two complementary sources. The Gaussian Mixture features summarise the fitted
mixture itself, namely the sorted component means, mixing weights and standard deviations, together
with shape statistics of the extracted lesion mask such as its area ratio, centroid position, spatial
spread, mean intensity and circularity. These capture how intensity is distributed in the slice and
where the candidate abnormality lies, which is exactly the information the hybrid pipeline supplies to
the network, but expressed as a small vector rather than as an image. Principal component scores
computed on downscaled grayscale images are appended to provide a compact description of global
anatomical appearance that the mixture descriptors do not contain.

Using the same unsupervised machinery for the classical models and for the hybrid pipeline is
deliberate: it isolates the contribution of convolutional representation learning, since both branches
have access to the same unsupervised information and differ only in how they consume it.

Feature extraction fits one mixture per image, so the training set is subsampled through
`limit_per_class` to keep the runtime of this notebook reasonable. The test set is used in full so
that every row of the comparison table is computed on identical test samples.


In [ ]:
CLASSICAL_TRAIN_LIMIT_PER_CLASS = 250

x_classical_raw, y_classical_train, _ = data_loader.load_split(
    config.TRAIN_DIR,
    image_size=config.IMAGE_SIZE,
    grayscale=False,
    limit_per_class=CLASSICAL_TRAIN_LIMIT_PER_CLASS,
    show_progress=True,
)
x_classical_train = preprocessing.preprocess_batch(x_classical_raw, image_size=config.IMAGE_SIZE)

print('Classical training subsample:', x_classical_train.shape)

feature_start = time.time()

train_gmm_features = gmm_segmentation.extract_gmm_features_batch(x_classical_train)
test_gmm_features = gmm_segmentation.extract_gmm_features_batch(x_test)

# PCA is fitted on the training subsample only and then applied to the test set. Fitting it on
# the pooled data would leak information about the test distribution into the representation.
pca, train_pca_scores = gmm_segmentation.fit_pca(x_classical_train)
test_pca_scores = gmm_segmentation.transform_pca(pca, x_test)

x_classical_features_train = np.hstack([train_gmm_features, train_pca_scores])
x_classical_features_test = np.hstack([test_gmm_features, test_pca_scores])

print('Feature construction completed in {0:.1f} seconds.'.format(time.time() - feature_start))
print('Training design matrix:', x_classical_features_train.shape)
print('Test design matrix:    ', x_classical_features_test.shape)
print('Variance explained by the retained components: {0:.4f}'.format(
    float(pca.explained_variance_ratio_.sum())
))

classical_results = classical_models.train_classical_models(
    x_classical_features_train,
    y_classical_train,
    x_classical_features_test,
    y_test,
)

classical_metric_rows = []
classical_display_names = {
    'logistic_regression': 'Logistic regression',
    'naive_bayes': 'Gaussian naive Bayes',
    'random_forest': 'Random forest',
    'svm': 'Support vector machine',
    'stacking_ensemble': 'Stacking ensemble',
}

for name, result in classical_results.items():
    display_name = classical_display_names.get(name, name)
    classical_metric_rows.append(
        evaluate.compute_metrics(y_test, result.predictions, display_name)
    )
    print('{0:<24} accuracy {1:.4f}  weighted F1 {2:.4f}'.format(
        display_name, result.accuracy, result.f1_weighted
    ))


In [ ]:
baseline_metrics = evaluate.compute_metrics(y_test, baseline_predictions, 'CNN baseline')
hybrid_metrics = evaluate.compute_metrics(y_test, hybrid_predictions, 'GMM and CNN hybrid')

comparison_table = evaluate.build_comparison_table(
    [baseline_metrics, hybrid_metrics] + classical_metric_rows
)

print('Consolidated comparison on the held out test set, sorted by weighted F1')
display(comparison_table.round(4))

comparison_path = config.METRICS_DIR / 'final_comparison.csv'
comparison_table.to_csv(comparison_path, index=False)
print('Comparison table written to', comparison_path)

best_row = comparison_table.iloc[0]
print('Highest weighted F1: {0} at {1:.4f}'.format(best_row['model'], best_row['f1_weighted']))


In [ ]:
figure = evaluate.plot_model_comparison(
    comparison_table,
    metrics=('accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted'),
    title='Model comparison on the held out test set',
    save_path=config.FIGURES_DIR / 'model_comparison.png',
)
plt.show()

# Macro averages are plotted separately because they weight every class equally and therefore
# reveal failures on the smaller classes that the weighted averages can conceal.
figure = evaluate.plot_model_comparison(
    comparison_table,
    metrics=('f1_macro', 'recall_macro', 'cohen_kappa', 'matthews_corrcoef'),
    title='Model comparison, macro averaged and chance corrected measures',
    save_path=config.FIGURES_DIR / 'model_comparison_macro.png',
)
plt.show()


## Per class behaviour

An aggregate score is a weighted average over classes, so a model can hold the higher average while
performing worse on an individual class. This happens whenever a change shifts the decision boundary
in a way that benefits the larger classes at the expense of a smaller one: the gain is multiplied by a
large support and the loss by a small one, and the weighted average improves even though one class has
deteriorated.

In a diagnostic application that trade is not automatically acceptable. A pipeline that raises overall
F1 while reducing recall on meningioma has increased the number of missed meningiomas, and no
improvement in the average compensates a patient whose lesion was not detected. The per class
comparison below is therefore read alongside the aggregate table rather than after it, and the
clinically weighted summary computed later in this notebook makes the same point in terms of the error
types that carry the clinical cost.


In [ ]:
baseline_per_class = evaluate.per_class_metrics(y_test, baseline_predictions)
baseline_per_class['model'] = 'CNN baseline'

hybrid_per_class = evaluate.per_class_metrics(y_test, hybrid_predictions)
hybrid_per_class['model'] = 'GMM and CNN hybrid'

per_class_long = pd.concat([baseline_per_class, hybrid_per_class], ignore_index=True)
display(per_class_long)

wide_f1 = per_class_long.pivot(index='class', columns='model', values='f1_score')
wide_f1['difference'] = wide_f1['GMM and CNN hybrid'] - wide_f1['CNN baseline']
print('Per class F1, hybrid minus baseline')
display(wide_f1.round(4))

figure, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=per_class_long, x='class', y='f1_score', hue='model', ax=ax)
ax.set_ylim(0, 1.05)
ax.set_xlabel('Class')
ax.set_ylabel('F1 score')
ax.set_title('Per class F1 score by model')
ax.legend(title='Model', loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)
figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'per_class_f1_comparison.png')
plt.show()


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(15, 5.8))

evaluate.plot_confusion_matrix(
    y_test,
    baseline_predictions,
    class_labels=config.CLASS_LABELS,
    title='CNN baseline, row normalised',
    normalise=True,
    ax=axes[0],
)

evaluate.plot_confusion_matrix(
    y_test,
    hybrid_predictions,
    class_labels=config.CLASS_LABELS,
    title='GMM and CNN hybrid, row normalised',
    normalise=True,
    ax=axes[1],
)

figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'confusion_matrix_comparison.png')
plt.show()

# A cell by cell difference localises the change precisely. Reading the two heatmaps side by
# side is unreliable for differences of a few percentage points.
baseline_matrix = confusion_matrix(y_test, baseline_predictions)
hybrid_matrix = confusion_matrix(y_test, hybrid_predictions)

baseline_rows = baseline_matrix.sum(axis=1, keepdims=True)
baseline_rows[baseline_rows == 0] = 1
hybrid_rows = hybrid_matrix.sum(axis=1, keepdims=True)
hybrid_rows[hybrid_rows == 0] = 1

baseline_normalised = baseline_matrix / baseline_rows
hybrid_normalised = hybrid_matrix / hybrid_rows

off_diagonal_changes = []
for true_index in range(config.NUM_CLASSES):
    for predicted_index in range(config.NUM_CLASSES):
        if true_index == predicted_index:
            continue
        change = hybrid_normalised[true_index, predicted_index] - baseline_normalised[true_index, predicted_index]
        off_diagonal_changes.append({
            'true': config.CLASS_LABELS[true_index],
            'predicted': config.CLASS_LABELS[predicted_index],
            'baseline_rate': round(float(baseline_normalised[true_index, predicted_index]), 4),
            'hybrid_rate': round(float(hybrid_normalised[true_index, predicted_index]), 4),
            'change': round(float(change), 4),
            'count_change': int(hybrid_matrix[true_index, predicted_index] - baseline_matrix[true_index, predicted_index]),
        })

changes_frame = pd.DataFrame(off_diagonal_changes).sort_values('change').reset_index(drop=True)

improved = changes_frame[changes_frame['change'] < 0]
worsened = changes_frame[changes_frame['change'] > 0]

print('Off diagonal cells reduced by the hybrid pipeline')
if len(improved):
    for _, row in improved.iterrows():
        print('  {0} predicted as {1}: {2:.4f} to {3:.4f}, change {4:+.4f} ({5:+d} images)'.format(
            row['true'], row['predicted'], row['baseline_rate'], row['hybrid_rate'],
            row['change'], row['count_change'],
        ))
else:
    print('  none')

print()
print('Off diagonal cells increased by the hybrid pipeline')
if len(worsened):
    for _, row in worsened.iterrows():
        print('  {0} predicted as {1}: {2:.4f} to {3:.4f}, change {4:+.4f} ({5:+d} images)'.format(
            row['true'], row['predicted'], row['baseline_rate'], row['hybrid_rate'],
            row['change'], row['count_change'],
        ))
else:
    print('  none')


In [ ]:
def macro_average_roc(y_true, y_score, n_classes=config.NUM_CLASSES):
    """Interpolate the one versus rest curves onto a common grid and average them.

    Averaging the curves rather than the scalar areas produces a single curve that can be
    plotted, which makes the comparison between the two models visible at every operating
    point rather than only at the summary statistic.
    """
    y_binary = label_binarize(y_true, classes=list(range(n_classes)))
    grid = np.linspace(0.0, 1.0, 201)
    interpolated = np.zeros_like(grid)

    for class_index in range(n_classes):
        false_positive_rate, true_positive_rate, _ = roc_curve(
            y_binary[:, class_index], y_score[:, class_index]
        )
        interpolated += np.interp(grid, false_positive_rate, true_positive_rate)

    interpolated /= n_classes
    return grid, interpolated, float(sklearn_auc(grid, interpolated))


baseline_grid, baseline_curve, baseline_macro_auc = macro_average_roc(y_test, baseline_probabilities)
hybrid_grid, hybrid_curve, hybrid_macro_auc = macro_average_roc(y_test, hybrid_probabilities)

figure, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].plot(baseline_grid, baseline_curve, linewidth=2,
             label='CNN baseline (AUC = {0:.4f})'.format(baseline_macro_auc))
axes[0].plot(hybrid_grid, hybrid_curve, linewidth=2,
             label='GMM and CNN hybrid (AUC = {0:.4f})'.format(hybrid_macro_auc))
axes[0].plot([0, 1], [0, 1], linestyle='--', color='grey', linewidth=1, label='Chance')
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].set_title('Macro averaged ROC, both convolutional models')
axes[0].legend(loc='lower right', fontsize=9)

_, hybrid_class_auc = evaluate.plot_roc_curves(
    y_test,
    hybrid_probabilities,
    class_labels=config.CLASS_LABELS,
    title='Hybrid model, per class ROC',
    ax=axes[1],
)

figure.tight_layout()
figure.savefig(config.FIGURES_DIR / 'roc_comparison.png')
plt.show()

# The per class areas for the baseline are needed for the table below but its per class panel was
# already saved in notebook 03, so the auxiliary figure is closed rather than displayed.
auxiliary_figure, auxiliary_ax = plt.subplots(figsize=(6, 5))
_, baseline_class_auc = evaluate.plot_roc_curves(
    y_test,
    baseline_probabilities,
    class_labels=config.CLASS_LABELS,
    ax=auxiliary_ax,
)
plt.close(auxiliary_figure)

auc_frame = pd.DataFrame({
    'CNN baseline': baseline_class_auc,
    'GMM and CNN hybrid': hybrid_class_auc,
})
auc_frame['difference'] = auc_frame['GMM and CNN hybrid'] - auc_frame['CNN baseline']
print('Area under the curve by class')
display(auc_frame.round(4))


## Statistical significance

The two convolutional models differ in accuracy by some amount. That difference is not by itself
evidence that one pipeline is better, because the test set is a finite sample: resampling it would
produce a slightly different value for both models, and a small gap can be entirely accounted for by
that variation.

An unpaired comparison of the two accuracy figures would discard the most useful property of this
experiment, namely that both models were evaluated on exactly the same images in exactly the same
order. The errors of the two models are therefore strongly dependent, since a slice that is genuinely
ambiguous tends to be misclassified by both. A test that assumed independence would overstate the
variance of the difference and would be unnecessarily conservative. A paired test is required.

McNemar's test is the standard paired test for two classifiers on a single test set. It arranges the
samples into a two by two contingency table according to whether each model was correct, and then
examines only the two discordant cells.

The concordant cells, where both models are correct or both are wrong, carry no information about
which model is better and are excluded from the statistic. The discordant cells are the evidence: the
count of samples that only the baseline classified correctly, and the count that only the hybrid
classified correctly. Under the null hypothesis that the two models have the same error rate, each
discordant sample is equally likely to fall into either cell, so the two counts should be
approximately equal. The test measures how improbable the observed imbalance is under that
assumption. The implementation uses the exact binomial form when the total number of discordant pairs
is below twenty five, where the chi squared approximation is unreliable, and the chi squared form with
a continuity correction otherwise.

A significant result indicates that the two models make systematically different errors, not that the
difference is clinically important. Statistical significance and practical relevance are separate
judgements, and both are addressed below.


In [ ]:
result = evaluate.mcnemar_test(y_test, baseline_predictions, hybrid_predictions)

print("McNemar's test, model A is the CNN baseline and model B is the GMM and CNN hybrid")
print('  Both correct:                 {0}'.format(result['both_correct']))
print('  Only the baseline correct:    {0}'.format(result['only_model_a_correct']))
print('  Only the hybrid correct:      {0}'.format(result['only_model_b_correct']))
print('  Both incorrect:               {0}'.format(result['both_incorrect']))
print('  Discordant pairs:             {0}'.format(
    result['only_model_a_correct'] + result['only_model_b_correct']
))
print('  Test used:                    {0}'.format(result['test_used']))
print('  Statistic:                    {0:.4f}'.format(result['statistic']))
print('  p value:                      {0:.6f}'.format(result['p_value']))
print('  Significant at 0.05:          {0}'.format(result['significant_at_0.05']))

print()
accuracy_difference = hybrid_metrics['accuracy'] - baseline_metrics['accuracy']

if result['significant_at_0.05']:
    better = 'hybrid' if result['only_model_b_correct'] > result['only_model_a_correct'] else 'baseline'
    print(
        'Interpretation: the p value of {0:.6f} is below the 0.05 threshold, so the null hypothesis '
        'that the two pipelines have the same error rate is rejected. The {1} pipeline is correct on '
        'significantly more of the discordant test samples, and the accuracy difference of {2:+.4f} '
        'is unlikely to be the result of sampling variation alone.'.format(
            result['p_value'], better, accuracy_difference
        )
    )
else:
    print(
        'Interpretation: the p value of {0:.6f} is above the 0.05 threshold, so the null hypothesis '
        'that the two pipelines have the same error rate is not rejected. The observed accuracy '
        'difference of {1:+.4f} is consistent with sampling variation on a test set of this size, '
        'and the evidence does not support a claim that either pipeline is better.'.format(
            result['p_value'], accuracy_difference
        )
    )

evaluate.save_metrics(result, 'mcnemar_baseline_vs_hybrid.json')


In [ ]:
# A confidence interval expresses the same uncertainty as the significance test in the units of
# the metric itself, which is easier to interpret than a p value when reporting results.
interval_rows = []
for model_name, model_predictions in (
    ('CNN baseline', baseline_predictions),
    ('GMM and CNN hybrid', hybrid_predictions),
):
    for metric_name in ('accuracy', 'f1_weighted'):
        interval = evaluate.bootstrap_confidence_interval(
            y_test,
            model_predictions,
            metric=metric_name,
            n_iterations=2000,
            confidence=0.95,
        )
        interval_rows.append({
            'model': model_name,
            'metric': interval['metric'],
            'point_estimate': round(interval['point_estimate'], 4),
            'lower_bound': round(interval['lower_bound'], 4),
            'upper_bound': round(interval['upper_bound'], 4),
            'interval_width': round(interval['upper_bound'] - interval['lower_bound'], 4),
        })

interval_frame = pd.DataFrame(interval_rows)
print('Bootstrap confidence intervals, two thousand resamples at the ninety five percent level')
display(interval_frame)

interval_path = config.METRICS_DIR / 'bootstrap_intervals.csv'
interval_frame.to_csv(interval_path, index=False)
print('Written to', interval_path)


## Disagreement analysis

The McNemar table reports how many samples lie in each discordant cell but says nothing about what
those samples look like. Examining them turns a statistic into an explanation.

The cases on which the two models disagree are precisely the cases where the segmentation stage
changed the decision, so they are the most informative subset of the test set for understanding what
the Gaussian Mixture prior actually contributes. If the images that only the hybrid model classifies
correctly are those in which the lesion is small or peripheral, the prior is doing the work it was
designed to do. If instead the images that only the baseline classifies correctly are those in which
the mask is degenerate, covering the skull or collapsing to nothing, then the failure mode of the
hybrid pipeline is a failure of the unsupervised stage rather than of the classifier, which is a
different problem with a different remedy.


In [ ]:
disagreement_indices = np.flatnonzero(baseline_predictions != hybrid_predictions)

baseline_correct = baseline_predictions == y_test
hybrid_correct = hybrid_predictions == y_test

print('Test samples where the two models disagree: {0} of {1} ({2:.2%})'.format(
    len(disagreement_indices), len(y_test), len(disagreement_indices) / len(y_test)
))
print('  Only the baseline correct: {0}'.format(
    int(np.sum(baseline_correct[disagreement_indices]))
))
print('  Only the hybrid correct:   {0}'.format(
    int(np.sum(hybrid_correct[disagreement_indices]))
))
print('  Both incorrect but differently: {0}'.format(
    int(np.sum(~baseline_correct[disagreement_indices] & ~hybrid_correct[disagreement_indices]))
))

if len(disagreement_indices):
    disagreement_frame = pd.DataFrame({
        'true': [config.CLASS_LABELS[int(label)] for label in y_test[disagreement_indices]],
        'baseline_prediction': [
            config.CLASS_LABELS[int(label)] for label in baseline_predictions[disagreement_indices]
        ],
        'hybrid_prediction': [
            config.CLASS_LABELS[int(label)] for label in hybrid_predictions[disagreement_indices]
        ],
    })
    print()
    print('Most frequent disagreement patterns')
    display(disagreement_frame.value_counts().head(8).to_frame('count'))

    rng = np.random.default_rng(config.RANDOM_SEED)
    selection_size = min(8, len(disagreement_indices))
    selected = rng.choice(disagreement_indices, size=selection_size, replace=False)

    figure, axes = plt.subplots(2, 4, figsize=(16, 9))
    for axis in axes.ravel():
        axis.axis('off')

    for axis, index in zip(axes.ravel(), selected):
        axis.imshow(np.clip(x_test[index], 0.0, 1.0))
        axis.set_title(
            'True: {0}\nBaseline: {1} ({2:.2f})\nHybrid: {3} ({4:.2f})'.format(
                config.CLASS_LABELS[int(y_test[index])],
                config.CLASS_LABELS[int(baseline_predictions[index])],
                float(baseline_probabilities[index].max()),
                config.CLASS_LABELS[int(hybrid_predictions[index])],
                float(hybrid_probabilities[index].max()),
            ),
            fontsize=10,
        )

    figure.suptitle('Test images on which the two convolutional models disagree', fontsize=14)
    figure.tight_layout()
    figure.savefig(config.FIGURES_DIR / 'model_disagreement_examples.png')
    plt.show()
else:
    print('The two models agree on every test sample, so there is nothing to display.')


## Efficiency considerations

Accuracy is not the only criterion by which a candidate system is judged. The two convolutional models
have identical parameter counts and therefore identical forward pass costs, but the pipelines around
them do not cost the same.

The hybrid pipeline must fit a Gaussian Mixture Model to each incoming scan before the network can be
evaluated. Expectation Maximisation over roughly fifty thousand pixel intensities is orders of
magnitude more expensive than a single convolutional forward pass, and unlike training it cannot be
amortised: the cost is paid again for every scan presented to the deployed system. It also runs on the
processor rather than on the accelerator, so it does not benefit from the hardware that makes the
network fast, and it does not batch efficiently because each image requires its own fit.

Three consequences follow for deployment. The end to end latency of the hybrid system is dominated by
the segmentation stage rather than by inference. Throughput under concurrent load is limited by
processor cores rather than by accelerator memory. And the system acquires an additional failure mode,
because Expectation Maximisation may converge to a degenerate solution on an unusual scan, which the
baseline pipeline cannot do.

The table below places the measured costs beside the measured accuracy so that the trade can be stated
in concrete terms rather than asserted. A statistically significant but small improvement that
multiplies end to end latency by a large factor may be the wrong choice for a screening deployment,
while the same improvement may be entirely acceptable for an offline second reader that processes
studies in batch overnight.


In [ ]:
# The segmentation stage is timed on a small sample because it is slow; a handful of images is
# sufficient to establish its order of magnitude relative to the forward pass.
SEGMENTATION_TIMING_SAMPLE = 20

segmentation_start = time.time()
for image in x_test[:SEGMENTATION_TIMING_SAMPLE]:
    gmm_segmentation.transform_for_hybrid(image)
segmentation_latency_ms = 1000.0 * (time.time() - segmentation_start) / SEGMENTATION_TIMING_SAMPLE

print('Gaussian Mixture transformation: {0:.2f} ms per image'.format(segmentation_latency_ms))


def read_training_seconds(filename):
    """Recover the recorded training time, returning None when the file is absent."""
    try:
        return float(evaluate.load_metrics(filename)['training_seconds'])
    except (FileNotFoundError, KeyError, TypeError):
        return None


baseline_training_seconds = read_training_seconds('baseline_history.json')
hybrid_training_seconds = read_training_seconds('hybrid_history.json')

efficiency_frame = pd.DataFrame([
    {
        'model': 'CNN baseline',
        'parameters': int(baseline_model.count_params()),
        'training_seconds': round(baseline_training_seconds, 1) if baseline_training_seconds else None,
        'preprocessing_ms_per_image': 0.0,
        'inference_ms_per_image': round(baseline_latency_ms, 2),
        'end_to_end_ms_per_image': round(baseline_latency_ms, 2),
        'accuracy': round(baseline_metrics['accuracy'], 4),
        'f1_weighted': round(baseline_metrics['f1_weighted'], 4),
    },
    {
        'model': 'GMM and CNN hybrid',
        'parameters': int(hybrid_model.count_params()),
        'training_seconds': round(hybrid_training_seconds, 1) if hybrid_training_seconds else None,
        'preprocessing_ms_per_image': round(segmentation_latency_ms, 2),
        'inference_ms_per_image': round(hybrid_network_latency_ms, 2),
        'end_to_end_ms_per_image': round(segmentation_latency_ms + hybrid_network_latency_ms, 2),
        'accuracy': round(hybrid_metrics['accuracy'], 4),
        'f1_weighted': round(hybrid_metrics['f1_weighted'], 4),
    },
])

print()
print('Cost and performance side by side')
display(efficiency_frame)

efficiency_path = config.METRICS_DIR / 'efficiency_comparison.csv'
efficiency_frame.to_csv(efficiency_path, index=False)

latency_ratio = (
    (segmentation_latency_ms + hybrid_network_latency_ms) / baseline_latency_ms
    if baseline_latency_ms > 0 else float('nan')
)
print('The hybrid pipeline costs {0:.1f} times the end to end latency of the baseline.'.format(
    latency_ratio
))
print('Written to', efficiency_path)


In [ ]:
# Weighted F1 is used for the selection because it balances precision against recall and accounts
# for the mild class imbalance, whereas accuracy alone would favour performance on the larger
# classes. The selection rule is fixed in advance so that it cannot be adjusted to favour a
# preferred outcome.
candidates = {
    'CNN baseline': (baseline_model, baseline_metrics),
    'GMM and CNN hybrid': (hybrid_model, hybrid_metrics),
}

selected_name = max(candidates, key=lambda name: candidates[name][1]['f1_weighted'])
selected_model, selected_metrics = candidates[selected_name]

best_model_path = config.MODELS_DIR / config.BEST_MODEL_NAME
selected_model.save(best_model_path)

selection_record = {
    'selected_model': selected_name,
    'selection_criterion': 'weighted F1 on the held out test set',
    'accuracy': round(selected_metrics['accuracy'], 4),
    'f1_weighted': round(selected_metrics['f1_weighted'], 4),
    'f1_macro': round(selected_metrics['f1_macro'], 4),
    'requires_gmm_preprocessing': selected_name == 'GMM and CNN hybrid',
    'mcnemar_p_value': round(result['p_value'], 6),
    'mcnemar_significant_at_0.05': bool(result['significant_at_0.05']),
    'saved_to': str(best_model_path),
}
evaluate.save_metrics(selection_record, 'model_selection.json')

print('Selected model: {0}'.format(selected_name))
print('  Weighted F1: {0:.4f}'.format(selected_metrics['f1_weighted']))
print('  Accuracy:    {0:.4f}'.format(selected_metrics['accuracy']))
print('  Saved to:    {0}'.format(best_model_path))
if selection_record['requires_gmm_preprocessing']:
    print('  The deployed application must apply gmm_segmentation.transform_for_hybrid before inference.')
else:
    print('  The deployed application needs only the standard preprocessing chain before inference.')
print()
print(config.MEDICAL_DISCLAIMER)


## Conclusions, limitations and future work

### Headline comparison

The consolidated table written to `results/metrics/final_comparison.csv` ranks every model evaluated in
this project by weighted F1 on the held out test set. The two convolutional pipelines occupy the top of
that ranking, and the classical feature baselines, which receive the same unsupervised Gaussian Mixture
and Principal Component information but consume it as a short vector rather than as an image, fall
clearly below them. That gap is the quantitative justification for using representation learning on
this task: the engineered descriptors capture intensity distribution and coarse lesion geometry but not
the textural detail that separates one tumour type from another.

The comparison central to the research question is between the baseline convolutional network and the
hybrid pipeline, and it is settled by the three quantities printed above rather than by assertion. The
difference in weighted F1 is read from the per class and comparison tables; the bootstrap intervals
show whether that difference is larger than the sampling uncertainty of a test set of this size; and
McNemar's test, together with the interpretation sentence printed by the significance cell, states
whether the null hypothesis of equal error rates is rejected at the five percent level. The model
selection cell applies the pre-specified rule to the resulting numbers and records the outcome in
`results/metrics/model_selection.json`, and the efficiency table states the cost at which any
improvement is obtained. Where the significance test does not reject the null hypothesis, the correct
conclusion is that this experiment provides no evidence that Gaussian Mixture pre-segmentation improves
classification, which is an informative result in its own right given the additional computational cost
the stage imposes.

### Limitations

The study rests on a single public dataset. All training and all evaluation derive from the same Kaggle
collection, which was assembled from a limited number of sources with their own scanner models,
acquisition protocols and patient population. Performance measured within that collection is an upper
bound on performance elsewhere, and the magnitude of the drop on data from a different institution
cannot be estimated from these experiments.

The models operate on two dimensional slices rather than on volumes. A radiologist reads a tumour as a
three dimensional structure across contiguous slices, and judges its extent, its relation to
surrounding anatomy and its growth pattern from that context. A single slice discards it. Some of the
residual confusion between glioma and meningioma is a direct consequence, because the distinction rests
substantially on whether the lesion arises within the parenchyma or from the meninges, which is a
volumetric relationship.

There is no external validation. No independent cohort from a different institution or scanner was
available, so generalisation beyond the source distribution remains untested.

There is no radiologist adjudication. The class labels were accepted as distributed, their provenance
and inter-rater agreement are unknown, and no clinician reviewed the model outputs. The reported
accuracy is therefore agreement with the dataset labels, not agreement with a verified clinical ground
truth.

The label space contains four classes. Real neuroradiological practice encompasses many further tumour
types, metastases, post-surgical change, infection and vascular lesions. A scan containing any of these
would be forced into one of the four available categories, with no mechanism for abstention, and the
confident misclassification that would result is a serious limitation for any screening use.

Two further constraints apply. The segmentation stage is unsupervised and was never evaluated against
expert lesion annotations, so the Grad-CAM overlap reported in notebook 04 measures agreement between
two automatic procedures rather than agreement with clinical truth. And the classical baselines were
trained on a subsample of the training data for runtime reasons, so their scores are a conservative
estimate of what those models could achieve with the full set.

### Future work

Extending the input from slices to three dimensional volumes is the most direct way to recover the
spatial context the current formulation discards, and it is the change most likely to reduce the
dominant confusions.

Fusing multiple magnetic resonance sequences, for example T1 weighted, T1 weighted with contrast, T2
weighted and FLAIR, would supply the complementary tissue contrast that radiologists rely upon and that
a single sequence cannot provide.

External validation on an independent cohort acquired at a different institution is a precondition for
any claim of generalisation and should precede any further architectural refinement.

Uncertainty calibration deserves explicit attention. Softmax outputs are not calibrated probabilities,
and a deployed system must be able to express low confidence so that ambiguous cases are escalated to a
human reader rather than silently assigned a class. Temperature scaling, deep ensembles or Monte Carlo
dropout are the standard options, and each would also provide a basis for an abstention mechanism.

A prospective clinical study, in which the system operates alongside reporting radiologists and its
influence on reading time and diagnostic agreement is measured, is the only way to establish whether
any of the measured accuracy translates into clinical value.

### Statement

This system is an academic prototype developed for the DS 4889 Machine Learning course at United
International University. It is not a certified medical device, it has not undergone regulatory
assessment or clinical validation, and it must not be used for diagnosis or for any decision affecting
patient care. All findings require confirmation by a qualified radiologist.
